In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_data
from src.features import build_features, COLUMN_ORDER
from src.models import RFCModel, XGBModel

In [3]:

df_raw = load_data('EEM')
X_raw = build_features(df_raw)

df_spy = X_raw.copy()

ma50  = df_raw['Close'].rolling(50).mean()
ma200 = df_raw['Close'].rolling(200).mean()
gc    = (ma50 > ma200).astype(int)
gc_clean = gc.reset_index(drop=True)

transition = np.zeros(len(df_spy), dtype=int)
label = gc_clean.iloc[0]
for i in range(1, len(gc_clean)):
    if gc_clean.iloc[i] != label:
        label = gc_clean.iloc[i]
        start = max(0, i - 30)
        transition[start:i] = 1

df_spy['Transition'] = transition

df_spy = df_spy.dropna().reset_index(drop=True)


X = df_spy[COLUMN_ORDER]
y = df_spy['Transition']

split = int(len(df_spy) * 0.75)

X_train = X.iloc[:split]
X_test  = X.iloc[split:]
y_train = y.iloc[:split]
y_test  = y.iloc[split:]

print(f"Train size: {len(X_train)}  |  positives: {y_train.sum()} ({y_train.mean()*100:.1f}%)")
print(f"Test size:  {len(X_test)}   |  positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
X_train.head(500)

Train size: 3946  |  positives: 751 (19.0%)
Test size:  1316   |  positives: 206 (15.7%)


,Return,Volatility,Cumulated_Return_5d,RSI14,Volume_ROC,ATR,VIX_spike,Distance_GC,MA_velocity,MA50_slope,Distance_normalized,MA_cross_momentum
0,-0.008982,0.010056,-0.031144,35.270595,-1.486698,0.168227,1.063226,0.180584,0.029775,0.027154,15.963973,-0.940642
1,-0.000302,0.010017,-0.015940,42.640889,26.207341,0.154234,1.052892,0.180822,0.034198,0.026764,16.027498,-0.906139
2,0.022786,0.011388,0.007862,51.584699,10.015898,0.170079,0.952759,0.181237,0.036198,0.026418,16.066949,-0.878080
3,0.014419,0.011888,0.022151,44.620010,-29.088278,0.160922,0.977098,0.181737,0.039310,0.025929,16.088302,-0.823523
4,-0.000874,0.011888,0.027006,43.478362,-47.214286,0.155366,0.951036,0.181952,0.039078,0.025691,16.085338,-0.798272
...,...,...,...,...,...,...,...,...,...,...,...,...
495,0.029063,0.016927,0.012176,67.164964,153.024674,0.456684,1.166422,0.131568,0.140464,0.034408,11.453763,-2.909787
496,0.015690,0.017098,0.046336,64.151037,-48.008088,0.428250,1.116501,0.133470,0.154415,0.033561,11.661015,-2.633897
497,0.000515,0.017121,0.047439,62.665999,-17.549013,0.435832,1.084368,0.135294,0.167629,0.032548,11.860078,-2.340378
498,0.023675,0.017217,0.038643,65.972974,62.915839,0.457474,1.049149,0.137550,0.177957,0.032202,12.095066,-2.177995


In [3]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# most_frequent : always predict the majority class
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print("=== Baseline (Most Frequent) ===")
print(classification_report(y_test, y_pred_dummy, 
      target_names=['No Transition', 'Transition'], zero_division=0))

#stratified: predict randomly according to class distriution
dummy_strat = DummyClassifier(strategy='stratified', random_state=42)
dummy_strat.fit(X_train, y_train)
y_pred_strat = dummy_strat.predict(X_test)

print("=== Baseline (Stratified Random) ===")
print(classification_report(y_test, y_pred_strat,
      target_names=['No Transition', 'Transition'], zero_division=0))

=== Baseline (Most Frequent) ===
               precision    recall  f1-score   support

No Transition       0.84      1.00      0.92      1110
   Transition       0.00      0.00      0.00       206

     accuracy                           0.84      1316
    macro avg       0.42      0.50      0.46      1316
 weighted avg       0.71      0.84      0.77      1316

=== Baseline (Stratified Random) ===
               precision    recall  f1-score   support

No Transition       0.84      0.81      0.82      1110
   Transition       0.16      0.20      0.18       206

     accuracy                           0.71      1316
    macro avg       0.50      0.50      0.50      1316
 weighted avg       0.74      0.71      0.72      1316



In [ ]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score
from scipy.stats import randint, uniform
from xgboost import XGBClassifier

f1_transition = make_scorer(f1_score, zero_division=0)

param_dist = {
    'n_estimators':     randint(200, 600),
    'max_depth':        randint(3, 8),
    'learning_rate':    uniform(0.01, 0.09),
    'subsample':        uniform(0.5, 0.4),
    'colsample_bytree': uniform(0.5, 0.4),
    'min_child_weight': randint(1, 10),      
    'gamma':            uniform(0, 0.3),     
}

tscv = TimeSeriesSplit(n_splits=5)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=30,   # early stopping
    verbosity=0,
    random_state=42
)

last_train_idx, last_val_idx = list(tscv.split(X_train))[-1]
eval_set = [(X_train.iloc[last_val_idx], y_train.iloc[last_val_idx])]

rs = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring=f1_transition,
    cv=tscv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=False          
)

rs.fit(X_train, y_train,
       eval_set=eval_set,
       verbose=False)

print("Best params :", rs.best_params_)
print("Best CV F1  :", round(rs.best_score_, 3))

best = rs.best_params_
xgb_tuned = XGBClassifier(
    **best,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    early_stopping_rounds=30,
    verbosity=0,
    random_state=42
)
xgb_tuned.fit(
    X_train.iloc[last_train_idx], y_train.iloc[last_train_idx],
    eval_set=eval_set,
    verbose=False
)

print(f"Best n_estimators (early stopping) : {xgb_tuned.best_iteration}")

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params : {'colsample_bytree': np.float64(0.551663766060598), 'gamma': np.float64(0.2862153081776167), 'learning_rate': np.float64(0.0645557171005792), 'max_depth': 3, 'min_child_weight': 1, 'n_estimators': 487, 'subsample': np.float64(0.5187585871164879)}
Best CV F1  : 0.699
Best n_estimators (early stopping) : 185


In [10]:
xgb = XGBModel(n_estimators=185,
            max_depth=3,
            learning_rate=0.0646,
            subsample=0.5187,
            colsample_bytree=0.551,
            gamma=0.286,
            min_child_weight=1,
            eval_metric='logloss',
            verbosity=0
        )
xgb.fit(X_train, y_train)

y_proba_xgb = xgb.predict_proba(X_test)
print("Threshold | Precision | Recall |  F1   | N_pred")
print("-" * 52)
for t in np.arange(0.3, 0.95, 0.05):
    pred = (y_proba_xgb >= t).astype(int)
    if pred.sum() > 0:
        p = precision_score(y_test, pred, zero_division=0)
        r = recall_score(y_test, pred, zero_division=0)
        f = f1_score(y_test, pred, zero_division=0)
        print(f"  {t:.2f}    |   {p:.3f}   |  {r:.3f} | {f:.3f} | {pred.sum()}")

Threshold | Precision | Recall |  F1   | N_pred
----------------------------------------------------
  0.30    |   0.488   |  0.898 | 0.632 | 379
  0.35    |   0.520   |  0.879 | 0.653 | 348
  0.40    |   0.548   |  0.859 | 0.669 | 323
  0.45    |   0.599   |  0.850 | 0.703 | 292
  0.50    |   0.622   |  0.830 | 0.711 | 275
  0.55    |   0.654   |  0.806 | 0.722 | 254
  0.60    |   0.668   |  0.762 | 0.712 | 235
  0.65    |   0.726   |  0.733 | 0.729 | 208
  0.70    |   0.769   |  0.694 | 0.730 | 186
  0.75    |   0.802   |  0.650 | 0.718 | 167
  0.80    |   0.850   |  0.578 | 0.688 | 140
  0.85    |   0.862   |  0.485 | 0.621 | 116
  0.90    |   0.911   |  0.398 | 0.554 | 90


In [14]:
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, f1_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint, uniform

f1_transition = make_scorer(f1_score, zero_division=0)

param_dist = {
    'n_estimators':      randint(200, 600),
    'max_depth':         randint(3, 10),
    'min_samples_split': randint(2, 20),
    'min_samples_leaf':  randint(1, 10),
    'max_features':      uniform(0.3, 0.5),   
}

tscv = TimeSeriesSplit(n_splits=5)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

rfc_base = RandomForestClassifier(
    class_weight='balanced',   
    random_state=42,
    n_jobs=-1
)

rs_rfc = RandomizedSearchCV(
    estimator=rfc_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring=f1_transition,
    cv=tscv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=True        
)

rs_rfc.fit(X_train, y_train)

print("Best params :", rs_rfc.best_params_)
print("Best CV F1  :", round(rs_rfc.best_score_, 3))

Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best params : {'max_depth': 8, 'max_features': np.float64(0.4093821097865351), 'min_samples_leaf': 8, 'min_samples_split': 10, 'n_estimators': 379}
Best CV F1  : 0.708


In [15]:
rfc = RFCModel(n_estimators=379,
            max_depth=8,
            min_samples_split=10,
            min_samples_leaf=8,
            max_features=0.409,
        )
rfc.fit(X_train, y_train)

y_proba_rfc = rfc.predict_proba(X_test)
print("Threshold | Precision | Recall |  F1   | N_pred")
print("-" * 52)
for t in np.arange(0.3, 0.95, 0.05):
    pred = (y_proba_rfc >= t).astype(int)
    if pred.sum() > 0:
        p = precision_score(y_test, pred, zero_division=0)
        r = recall_score(y_test, pred, zero_division=0)
        f = f1_score(y_test, pred, zero_division=0)
        print(f"  {t:.2f}    |   {p:.3f}   |  {r:.3f} | {f:.3f} | {pred.sum()}")

Threshold | Precision | Recall |  F1   | N_pred
----------------------------------------------------
  0.30    |   0.409   |  0.893 | 0.561 | 450
  0.35    |   0.448   |  0.893 | 0.596 | 411
  0.40    |   0.511   |  0.888 | 0.649 | 358
  0.45    |   0.549   |  0.869 | 0.673 | 326
  0.50    |   0.592   |  0.825 | 0.690 | 287
  0.55    |   0.660   |  0.801 | 0.724 | 250
  0.60    |   0.692   |  0.752 | 0.721 | 224
  0.65    |   0.767   |  0.704 | 0.734 | 189
  0.70    |   0.821   |  0.670 | 0.738 | 168
  0.75    |   0.878   |  0.524 | 0.657 | 123
  0.80    |   0.947   |  0.437 | 0.598 | 95
  0.85    |   0.965   |  0.398 | 0.564 | 85
  0.90    |   1.000   |  0.277 | 0.433 | 57


In [13]:
def compute_transition(df_raw):
    ma50  = df_raw['Close'].rolling(50).mean()
    ma200 = df_raw['Close'].rolling(200).mean()
    gc    = (ma50 > ma200).astype(int)
    transition = np.zeros(len(df_raw), dtype=int)
    label = gc.iloc[0]
    for i in range(1, len(gc)):
        if gc.iloc[i] != label:
            label = gc.iloc[i]
            start = max(0, i - 30)
            transition[start:i] = 1
    return pd.Series(transition, name='Transition')


def prepare_asset(ticker, split=0.75):
    df_raw = load_data(ticker)
    X_raw  = build_features(df_raw)
    df     = X_raw.copy()
    df['Transition'] = compute_transition(df_raw).values
    df     = df.dropna().reset_index(drop=True)
    split_idx = int(len(df) * split)
    X = df[COLUMN_ORDER]
    y = df['Transition']
    return X.iloc[split_idx:], y.iloc[split_idx:]


def validate_cross_assets(rfc_model, xgb_model, thresh_rfc, thresh_xgb, tickers):
    results = []
    for ticker in tickers:
        X_test, y_test = prepare_asset(ticker)
        print(f"\n{'='*50}")
        print(f"Asset: {ticker} | positives: {y_test.sum()} ({y_test.mean()*100:.1f}%)")

        for model, thresh, name in [
            (rfc_model, thresh_rfc, 'RFC'),
            (xgb_model, thresh_xgb, 'XGB')
        ]:
            proba = model.predict_proba(X_test)
            pred  = (proba >= thresh).astype(int)
            results.append({
                'Asset':     ticker,
                'Model':     name,
                'Precision': round(precision_score(y_test, pred, zero_division=0), 3),
                'Recall':    round(recall_score(y_test, pred, zero_division=0), 3),
                'F1':        round(f1_score(y_test, pred, zero_division=0), 3),
                'N_pred':    int(pred.sum())
            })

    df_results = pd.DataFrame(results)
    print("\n" + "="*60)
    print("CROSS-ASSET VALIDATION SUMMARY")
    print("="*60)
    print(df_results.to_string(index=False))
    print(f"\nMean F1 RFC: {df_results[df_results['Model']=='RFC']['F1'].mean():.3f}")
    print(f"Mean F1 XGB: {df_results[df_results['Model']=='XGB']['F1'].mean():.3f}")
    return df_results


results = validate_cross_assets(rfc, xgb, thresh_rfc=0.65, thresh_xgb=0.7, tickers=['SPY', 'DIA', 'QQQ'])


Asset: SPY | positives: 150 (9.9%)

Asset: DIA | positives: 157 (10.3%)

Asset: QQQ | positives: 132 (8.7%)

CROSS-ASSET VALIDATION SUMMARY
Asset Model  Precision  Recall    F1  N_pred
  SPY   RFC      0.798   0.687 0.738     129
  SPY   XGB      0.747   0.807 0.776     162
  DIA   RFC      0.675   0.675 0.675     157
  DIA   XGB      0.584   0.688 0.632     185
  QQQ   RFC      0.894   0.833 0.863     123
  QQQ   XGB      0.857   0.864 0.860     133

Mean F1 RFC: 0.759
Mean F1 XGB: 0.756


In [ ]:
results = validate_cross_assets(rfc, xgb, thresh_rfc=0.65, thresh_xgb=0.7, tickers=['GLD', 'TLT', 'USO', 'VNQ', 'URTH', 'EEM', 'EWJ', 'CAC']) 


Asset: GLD | positives: 273 (22.5%)

Asset: TLT | positives: 264 (19.4%)

Asset: USO | positives: 180 (16.0%)

Asset: VNQ | positives: 243 (19.9%)

Asset: URTH | positives: 60 (7.9%)

Asset: EEM | positives: 206 (15.7%)

Asset: EWJ | positives: 184 (12.1%)

Asset: CAC | positives: 210 (13.8%)

CROSS-ASSET VALIDATION SUMMARY
Asset Model  Precision  Recall    F1  N_pred
  GLD   RFC      0.887   0.890 0.888     274
  GLD   XGB      0.886   0.799 0.840     246
  TLT   RFC      0.668   0.716 0.691     283
  TLT   XGB      0.673   0.701 0.686     275
  USO   RFC      0.611   0.550 0.579     162
  USO   XGB      0.578   0.600 0.589     187
  VNQ   RFC      0.559   0.741 0.637     322
  VNQ   XGB      0.588   0.770 0.667     318
 URTH   RFC      0.626   0.950 0.755      91
 URTH   XGB      0.626   0.950 0.755      91
  EEM   RFC      0.767   0.704 0.734     189
  EEM   XGB      0.769   0.694 0.730     186
  EWJ   RFC      0.397   0.641 0.491     297
  EWJ   XGB      0.408   0.674 0.508     30